In [ ]:
#!/usr/bin/env python
# coding: utf-8

# In[1]:


import math
from typing import List, Optional, Tuple, Union
import torch
import torch.nn.functional as F
import torch.utils.checkpoint
from torch import nn
import os
import pandas as pd

from torch.utils.data import IterableDataset, Dataset
import json
import numpy as np
from transformers import  PreTrainedModel
from transformers.modeling_outputs import CausalLMOutputWithPast
from transformers import PretrainedConfig
from transformers import Trainer, TrainingArguments, AutoModelForCausalLM, AutoTokenizer, DefaultDataCollator, DataCollatorForTokenClassification, AutoConfig

class SFTDataset(Dataset):
    def __init__(self, data_path, tokenizer, max_seq_len):
        super().__init__()
        self.data_path = data_path
        self.tokenizer = tokenizer
        self.max_seq_len = max_seq_len
        self.padding_id = tokenizer.pad_token_id
        with open(self.data_path, 'r', encoding='utf-8') as f:
            self.data = json.load(f)

    def __len__(self):
        return len(self.data)    
    
    def __getitem__(self, index):
        line = self.data[index]
        # instruction_text = line['instruction']
        # input_text = line['input']
        # query = instruction_text + input_text
        query = line['query']
        output_text = line['response']
        answer = output_text + self.tokenizer.eos_token
        messages = []
        messages.append({'role': 'user', 'content': query})   
        prompt = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True) 
        
        prompt_input_ids = self.tokenizer.encode(prompt)
        answer_input_ids = self.tokenizer.encode(answer)
        
        input_ids = prompt_input_ids + answer_input_ids
        labels = [-100] * len(prompt_input_ids) + answer_input_ids
        attention_mask = [1] * len(input_ids)
        text_len = len(input_ids)
        
        if text_len > self.max_seq_len:
            input_ids = input_ids[:self.max_seq_len]
            labels = labels[:self.max_seq_len]
            attention_mask = attention_mask[:self.max_seq_len]
        else:
            input_ids = input_ids + [self.tokenizer.pad_token_id] * (self.max_seq_len - text_len)
            labels = labels + [-100] * (self.max_seq_len - text_len)
            attention_mask = attention_mask + [0] * (self.max_seq_len - text_len)
        
        # input_ids = input_ids[:-1]
        # labels = labels[1:]
        return {'input_ids': torch.tensor(input_ids), 'attention_mask':torch.tensor(attention_mask), 'labels': torch.tensor(labels)}
    


# In[2]:


# 计算前向kl散度
def compute_fkl(
        logits, 
        teacher_logits, 
        target, 
        padding_id,
        reduction="sum",
        temp = 1.0, 
        
    ):
        logits = logits / temp
        teacher_logits = teacher_logits / temp

        log_probs = torch.log_softmax(logits, -1, dtype=torch.float32)
        teacher_probs = torch.softmax(teacher_logits, -1, dtype=torch.float32)
        teacher_log_probs = torch.log_softmax(teacher_logits, -1, dtype=torch.float32)
        kl = (teacher_probs * (teacher_log_probs - log_probs)) 
        kl = kl.sum(-1)
        pad_mask = target.eq(padding_id)
        kl = kl.masked_fill_(pad_mask, 0.0)
        if reduction == "sum":
            
            kl = kl.sum(dim=1)
        
        elif reduction == "mean":
            kl = kl.sum(dim=1) / (~pad_mask).sum(dim=1)

        return kl


# In[3]:


from transformers import AutoModelForCausalLM, AutoTokenizer, DefaultDataCollator
from peft import LoraConfig, get_peft_model, TaskType
from peft import PeftModel
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import Trainer, TrainingArguments
from swanlab.integration.transformers import SwanLabCallback

# from dataset import SFTDataset
# from utils import compute_fkl, compute_rkl, compute_skewed_fkl, compute_skewed_rkl



class KGTrainer(Trainer):
    
    def __init__(
        self,
        model = None,
        teacher_model = None,
        if_use_entropy = False,
        args = None,
        data_collator = None, 
        train_dataset = None,
        eval_dataset = None,
        model_init = None, 
        compute_metrics = None, 
        callbacks = None,
        optimizers = (None, None), 
        preprocess_logits_for_metrics = None,
        # tokenizer=None,
    ):
        super().__init__(
            model=model,
            args=args,
            data_collator=data_collator,
            train_dataset=train_dataset,
            eval_dataset=eval_dataset,
            model_init=model_init,
            compute_metrics=compute_metrics,
            callbacks=callbacks,
            optimizers=optimizers,
            preprocess_logits_for_metrics=preprocess_logits_for_metrics,
            # tokenizer=tokenizer
        )
        self.teacher_model = teacher_model
        self.if_use_entropy = if_use_entropy
        
    
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        
        outputs = model(**inputs)
        with torch.no_grad():
            teacher_outputs = self.teacher_model(**inputs)
        
        loss = outputs.loss
        logits = outputs.logits
        teacher_logits = teacher_outputs.logits
        
        # 如果教师模型和学生模型输出形状不匹配，对学生模型进行padding或对教师模型进行截断
        if logits.shape[-1] != teacher_logits.shape[-1]:
            teacher_logits = teacher_logits[:, :, :logits.shape[-1]]
        
        labels = inputs['labels']
        kl = compute_fkl(logits, teacher_logits, labels, padding_id=-100, temp=1.0).mean()
        
        if self.if_use_entropy:
            alpha = 0.01
            loss_total = alpha * kl + (1-alpha) * loss
            loss_total = alpha * kl + loss
        else:
            loss_total = loss
        
        return (loss_total, outputs) if return_outputs else loss_total


class BasicTrainer(Trainer):
    
    def __init__(
        self,
        model = None,
        # teacher_model = None,
        # if_use_entropy = False,
        args = None,
        data_collator = None, 
        train_dataset = None,
        eval_dataset = None,
        model_init = None, 
        compute_metrics = None, 
        callbacks = None,
        optimizers = (None, None), 
        preprocess_logits_for_metrics = None,
        # tokenizer=None,
    ):
        super().__init__(
            model=model,
            args=args,
            data_collator=data_collator,
            train_dataset=train_dataset,
            eval_dataset=eval_dataset,
            model_init=model_init,
            compute_metrics=compute_metrics,
            callbacks=callbacks,
            optimizers=optimizers,
            preprocess_logits_for_metrics=preprocess_logits_for_metrics,
            # tokenizer=tokenizer
        )
        # self.teacher_model = teacher_model
        # self.if_use_entropy = if_use_entropy
        
    
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        
        outputs = model(**inputs)
        
        loss = outputs.loss
        logits = outputs.logits
        
        loss_total = loss
        
        return (loss_total, outputs) if return_outputs else loss_total

# In[4]:


if __name__ == '__main__':
    
    # 学生模型
    model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-3B-Instruct")
    
    lora_config = LoraConfig(
        r=4,  
        lora_alpha=8,  
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        lora_dropout=0.1, 
        task_type=TaskType.CAUSAL_LM
    )
    
    # 使用lora方法训练
    model = get_peft_model(model, lora_config)
    model.cuda()
    print(model.print_trainable_parameters())
    print(f'Load Student Model Complete.\n')
    
    tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-3B-Instruct")
    
    # 教师模型，在给定数据上通过lora微调
    # teacher_model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-Math-7B-Instruct")

    # teacher_model.cuda()
    # teacher_model.eval()

    # print(f'Load Teacher Model Complete.\n')

    args = TrainingArguments(output_dir='./results_0223_exp5', 
                            num_train_epochs=4, 
                            do_train=True, 
                            per_device_train_batch_size=2,
                            per_device_eval_batch_size=2,
                            gradient_accumulation_steps=16,
                            logging_steps=100,             # 每 100 步记录一次训练损失
                            eval_strategy="steps",         # 将 epoch 改为 steps
                            eval_steps=100,                # 每 100 步进行一次验证
                            report_to='none',
                            save_strategy='steps',
                            save_steps=100,
                            save_total_limit=10,
                            bf16=True,
                            learning_rate=0.00006,
                            lr_scheduler_type='cosine',
                            dataloader_num_workers=1,
                            dataloader_pin_memory=True,
    )

    data_collator = DefaultDataCollator()
    train_dataset = SFTDataset('data/data_train_large_math7000.json', tokenizer=tokenizer, max_seq_len=1024)
    val_dataset = SFTDataset('data/data_valid_large_math7000.json', tokenizer=tokenizer, max_seq_len=1024)

    swanlab_callback = SwanLabCallback(
        project="qwen2.5-3b-distill",
        workspace="mtm22",
        experiment_name="SFT-0223-exp5" ,
        # experiment_id="cdhfh6fwq6x5qxftusmc6"
    )

    # trainer = KGTrainer(model=model,
    #                     teacher_model=teacher_model, 
    #                     if_use_entropy = False,
    #                     args=args, 
    #                     train_dataset=train_dataset, 
    #                     eval_dataset=val_dataset,
    #                     callbacks=[swanlab_callback],
    #                     # tokenizer=tokenizer, 
    #                     data_collator=data_collator)
    
    trainer = BasicTrainer(model=model,
                        args=args, 
                        train_dataset=train_dataset, 
                        eval_dataset=val_dataset,
                        callbacks=[swanlab_callback],
                        # tokenizer=tokenizer, 
                        data_collator=data_collator)

    # In[ ]:


    trainer.train(resume_from_checkpoint=False)
    trainer.save_model('./saves_0223_exp5')
    trainer.save_state()


⠸ Creating experiment...

swanlab: Tracking run with swanlab version 0.7.8

swanlab: Run data will be saved locally in 
/home/ha5083li/Downloads/llm_related/knowledge_distillation_llm/swanlog/run-20260221_154340-pa7xyovgzcrnsexq4ondl

swanlab: 👋 Hi mtm22,welcome to swanlab!

swanlab: Syncing run SFT-0221-exp4 to the cloud

swanlab: 🏠 View project at https://swanlab.cn/@mtm22/qwen2.5-3b-distill

swanlab: 🚀 View run at https://swanlab.cn/@mtm22/qwen2.5-3b-distill/runs/pa7xyovgzcrnsexq4ondl

Step,Training Loss,Validation Loss
100,4.995746,0.152011
200,4.316882,0.149969
300,4.222440,0.149169
400,4.235826,0.148851
500,4.231721,0.149039
600,4.043250,0.148469
700,3.993171,0.148514
800,4.132833,0.149282
900,3.897043,0.149798
1000,3.711049,0.149735


swanlab: swanlab api error ('Trace id: None', 'POST https://api.swanlab.cn/api/house/metrics', '522 Connect origin 
timed out')

In [1]:
import math
from typing import List, Optional, Tuple, Union
import torch
import torch.nn.functional as F
import torch.utils.checkpoint
from torch import nn
import os
import pandas as pd

from torch.utils.data import IterableDataset, Dataset
import json
import numpy as np
from transformers import  PreTrainedModel
from transformers.modeling_outputs import CausalLMOutputWithPast
from transformers import PretrainedConfig
from transformers import Trainer, TrainingArguments, AutoModelForCausalLM, AutoTokenizer, DefaultDataCollator, DataCollatorForTokenClassification, AutoConfig

class SFTDataset(Dataset):
    def __init__(self, data_path, tokenizer, max_seq_len):
        super().__init__()
        self.data_path = data_path
        self.tokenizer = tokenizer
        self.max_seq_len = max_seq_len
        self.padding_id = tokenizer.pad_token_id
        with open(self.data_path, 'r', encoding='utf-8') as f:
            self.data = json.load(f)

    def __len__(self):
        return len(self.data)    
    
    def __getitem__(self, index):
        line = self.data[index]
        # instruction_text = line['instruction']
        # input_text = line['input']
        # query = instruction_text + input_text
        query = line['query']
        output_text = line['response']
        answer = output_text + self.tokenizer.eos_token
        messages = []
        messages.append({'role': 'user', 'content': query})   
        prompt = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True) 
        
        prompt_input_ids = self.tokenizer.encode(prompt)
        answer_input_ids = self.tokenizer.encode(answer)
        
        input_ids = prompt_input_ids + answer_input_ids
        labels = [-100] * len(prompt_input_ids) + answer_input_ids
        attention_mask = [1] * len(input_ids)
        text_len = len(input_ids)
        
        if text_len > self.max_seq_len:
            input_ids = input_ids[:self.max_seq_len]
            labels = labels[:self.max_seq_len]
            attention_mask = attention_mask[:self.max_seq_len]
        else:
            input_ids = input_ids + [self.tokenizer.pad_token_id] * (self.max_seq_len - text_len)
            labels = labels + [-100] * (self.max_seq_len - text_len)
            attention_mask = attention_mask + [0] * (self.max_seq_len - text_len)
        
        # input_ids = input_ids[:-1]
        # labels = labels[1:]
        return {'input_ids': torch.tensor(input_ids), 'attention_mask':torch.tensor(attention_mask), 'labels': torch.tensor(labels)}
    

/home/ha5083li/miniconda3/envs/train_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

# 计算前向kl散度
def compute_fkl(
        logits, 
        teacher_logits, 
        target, 
        padding_id,
        reduction="sum",
        temp = 1.0, 
        
    ):
        logits = logits / temp
        teacher_logits = teacher_logits / temp

        log_probs = torch.log_softmax(logits, -1, dtype=torch.float32)
        teacher_probs = torch.softmax(teacher_logits, -1, dtype=torch.float32)
        teacher_log_probs = torch.log_softmax(teacher_logits, -1, dtype=torch.float32)
        kl = (teacher_probs * (teacher_log_probs - log_probs)) 
        kl = kl.sum(-1)
        pad_mask = target.eq(padding_id)
        kl = kl.masked_fill_(pad_mask, 0.0)
        if reduction == "sum":
            
            kl = kl.sum(dim=1)
        
        elif reduction == "mean":
            kl = kl.sum(dim=1) / (~pad_mask).sum(dim=1)

        return kl


In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer, DefaultDataCollator
from peft import LoraConfig, get_peft_model, TaskType
from peft import PeftModel
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import Trainer, TrainingArguments
from swanlab.integration.transformers import SwanLabCallback

# from dataset import SFTDataset
# from utils import compute_fkl, compute_rkl, compute_skewed_fkl, compute_skewed_rkl


class KGTrainer(Trainer):
    
    def __init__(
        self,
        model = None,
        teacher_model = None,
        if_use_entropy = False,
        args = None,
        data_collator = None, 
        train_dataset = None,
        eval_dataset = None,
        model_init = None, 
        compute_metrics = None, 
        callbacks = None,
        optimizers = (None, None), 
        preprocess_logits_for_metrics = None,
        # tokenizer=None,
    ):
        super().__init__(
            model=model,
            args=args,
            data_collator=data_collator,
            train_dataset=train_dataset,
            eval_dataset=eval_dataset,
            model_init=model_init,
            compute_metrics=compute_metrics,
            callbacks=callbacks,
            optimizers=optimizers,
            preprocess_logits_for_metrics=preprocess_logits_for_metrics,
            # tokenizer=tokenizer
        )
        self.teacher_model = teacher_model
        self.if_use_entropy = if_use_entropy
        
    
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        
        outputs = model(**inputs)
        with torch.no_grad():
            teacher_outputs = self.teacher_model(**inputs)
        
        loss = outputs.loss
        logits = outputs.logits
        teacher_logits = teacher_outputs.logits
        
        # 如果教师模型和学生模型输出形状不匹配，对学生模型进行padding或对教师模型进行截断
        if logits.shape[-1] != teacher_logits.shape[-1]:
            teacher_logits = teacher_logits[:, :, :logits.shape[-1]]
        
        labels = inputs['labels']
        kl = compute_fkl(logits, teacher_logits, labels, padding_id=-100, temp=1.0).mean()
        
        if self.if_use_entropy:
            alpha = 0.01
            loss_total = alpha * kl + (1-alpha) * loss
            loss_total = alpha * kl + loss
        else:
            loss_total = loss
        
        return (loss_total, outputs) if return_outputs else loss_total

In [4]:
if __name__ == '__main__':
    
    # 学生模型
    model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-3B-Instruct")
    
    lora_config = LoraConfig(
        r=4,  
        lora_alpha=16,  
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        lora_dropout=0.1, 
        task_type=TaskType.CAUSAL_LM
    )
    
    # 使用lora方法训练
    model = get_peft_model(model, lora_config)
    model.cuda()
    print(model.print_trainable_parameters())
    print(f'Load Student Model Complete.\n')
    
    tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-3B-Instruct")
    
    # 教师模型，在给定数据上通过lora微调
    teacher_model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-Math-7B-Instruct")
    # 是否加载lora模型
    # lora_path = 'qwen2.5_7b/lora/sft'
    # teacher_model = PeftModel.from_pretrained(teacher_model, lora_path)
    teacher_model.cuda()
    teacher_model.eval()

    print(f'Load Teacher Model Complete.\n')

    
      

Loading weights: 100%|██████████| 434/434 [00:05<00:00, 84.09it/s, Materializing param=model.norm.weight]                               


trainable params: 14,966,784 || all params: 3,100,905,472 || trainable%: 0.4827
None
Load Student Model Complete.



Loading weights: 100%|██████████| 339/339 [00:04<00:00, 70.47it/s, Materializing param=model.norm.weight]                               


Load Teacher Model Complete.



In [5]:
    args = TrainingArguments(output_dir='./results', 
                            num_train_epochs=10, 
                            do_train=True, 
                            per_device_train_batch_size=2,
                            per_device_eval_batch_size=2,
                            gradient_accumulation_steps=16,
                            logging_steps=100,             # 每 100 步记录一次训练损失
                            eval_strategy="steps",         # 将 epoch 改为 steps
                            eval_steps=100,                # 每 100 步进行一次验证
                            report_to='none',
                            save_strategy='steps',
                            save_total_limit=5,
                            bf16=True,
                            learning_rate=0.0001,
                            lr_scheduler_type='cosine',
                            dataloader_num_workers=1,
                            dataloader_pin_memory=True,
    )
    
    data_collator = DefaultDataCollator()
    train_dataset = SFTDataset('data_train_large.json', tokenizer=tokenizer, max_seq_len=1024)
    val_dataset = SFTDataset('data_valid_large.json', tokenizer=tokenizer, max_seq_len=1024)
    
    swanlab_callback = SwanLabCallback(
        project="qwen2.5-3b-distill",
        workspace="mtm22",
        experiment_name="SFT-0215-batch1_16" ,
        # experiment_id="cdhfh6fwq6x5qxftusmc6"
    )
    
    trainer = KGTrainer(model=model,
                        teacher_model=teacher_model, 
                        if_use_entropy = False,
                        args=args, 
                        train_dataset=train_dataset, 
                        eval_dataset=val_dataset,
                        callbacks=[swanlab_callback],
                        # tokenizer=tokenizer, 
                        data_collator=data_collator)

In [ ]:
    trainer.train(resume_from_checkpoint=False)
    trainer.save_model('./saves_datalarge')
    trainer.save_state()

/home/ha5083li/miniconda3/envs/train_env/lib/python3.10/site-packages/rich/live.py:231: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

swanlab: Tracking run with swanlab version 0.7.8

swanlab: Run data will be saved locally in 
/home/ha5083li/Downloads/llm_related/knowledge_distillation_llm/swanlog/run-20260215_220155-nuky2g5fgtw7yfudmy6y6

swanlab: 👋 Hi mtm22,welcome to swanlab!

swanlab: Syncing run SFT-0215-batch1_16 to the cloud

swanlab: 🏠 View project at https://swanlab.cn/@mtm22/qwen2.5-3b-distill

swanlab: 🚀 View run at https://swanlab.cn/@mtm22/qwen2.5-3b-distill/runs/nuky2g5fgtw7yfudmy6y6

Step,Training Loss,Validation Loss
100,4.652000,0.160417
200,4.473055,0.159986
300,4.400264,0.159811
400,4.132852,0.167861
500,3.218180,0.171707
600,3.205653,0.170937
700,3.234410,0.168117
800,2.785252,0.198646
900,2.139783,0.197377
1000,2.152282,0.198071
